# LoRe on PRISM — full pipeline + Rachel Freedman heatmap

Runs the real Facebook Research LoRe pipeline on the PRISM dataset using the Skywork-Reward-Llama-3.1-8B embedding model, then produces a heatmap of learned user weights for K=8 bases.

**Before running:** Runtime → Change runtime type → **A100 GPU**.  Then Runtime → Run all.

Total expected time on A100: ~45–75 min (model download ~5 min, embeddings ~30–60 min, training+plot ~5 min).

## 1. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > A100.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)
print('torch:', torch.__version__)

## 2. Clone LoRe repo

In [ ]:
%cd /content
!rm -rf LoRe
!git clone https://github.com/facebookresearch/LoRe.git
%cd /content/LoRe

## 3. Install missing deps

Colab already has torch, transformers, datasets, numpy, pandas, matplotlib. We just need a few extras.

In [ ]:
!pip install -q pydantic torchinfo safetensors accelerate sentencepiece

## 4. HuggingFace login

Run the cell, paste your `hf_...` token into the prompt that appears, hit Enter. Token will not be visible.

In [ ]:
from huggingface_hub import login
from getpass import getpass
login(token=getpass('HuggingFace token: '))

## 5. Patch repo scripts

- `prepare.py` imports `IPython.core.ultratb` and sets `sys.excepthook` — fine in IPython but noisy. Remove.
- The repo hardcodes `device = "cuda:0"` in multiple files. Colab uses `cuda` (single GPU). `cuda:0` resolves correctly here, so no patch needed — but we'll verify.

In [ ]:
# Remove the IPython ultratb hook from prepare.py (not needed, can cause issues in non-IPython contexts)
import re
with open('/content/LoRe/PRISM/prepare.py', 'r') as f:
    src = f.read()
src = re.sub(r'from IPython\.core import ultratb\nsys\.excepthook = ultratb\.FormattedTB\(call_pdb=1\)\n', '', src)
with open('/content/LoRe/PRISM/prepare.py', 'w') as f:
    f.write(src)
print('Patched prepare.py')

## 6. Run `prepare.py` — download PRISM, build train/test parquet

Downloads ~500MB of JSONL from HuggingFace. Takes a couple of minutes.

In [ ]:
%cd /content/LoRe/PRISM
!python prepare.py 2>&1 | tail -30

## 7. Generate Skywork embeddings (the long step)

Downloads the 16GB Skywork-Reward-Llama-3.1-8B model once, then runs ~16k forward passes to embed every (prompt, chosen) and (prompt, rejected) pair. On A100 this is roughly 30–60 min. The progress bar comes from `tqdm` inside the script.

If you're cost/time-conscious, set `SUBSET_USERS` below to e.g. 100 to embed only the first 100 users (~5–10 min). Set to `None` for the full run.

In [ ]:
SUBSET_USERS = None  # set to e.g. 100 for a fast pilot run, or None for full PRISM

if SUBSET_USERS is not None:
    # Subset the parquet files in place: keep only rows from the first SUBSET_USERS user_ids
    import pandas as pd, json
    with open('/content/LoRe/PRISM/data/prism/prism_split_ids_50.json') as f:
        split = json.load(f)
    keep_users = set(list(split['seen_user_ids'].keys())[:SUBSET_USERS]
                     + list(split['unseen_user_ids'].keys())[:max(20, SUBSET_USERS // 5)])
    for fp in ['/content/LoRe/PRISM/data/prism/train.parquet',
               '/content/LoRe/PRISM/data/prism/test.parquet']:
        df = pd.read_parquet(fp)
        before = len(df)
        df = df[df['extra_info'].apply(lambda x: x['user_id'] in keep_users)]
        df.to_parquet(fp)
        print(f'{fp}: {before} -> {len(df)} rows')
    print(f'Subsetted to {len(keep_users)} users')
else:
    print('Running full PRISM (no subset)')

In [ ]:
%cd /content/LoRe/PRISM
!python generate-prism-embeddings.py

## 8. Train LoRe basis (K=8) and extract V, W directly

Instead of running `train_basis.py` (which sweeps over many K and only prints accuracy), we call the underlying `solve_regularized_simplex` once at K=8 and keep the learned basis matrix V and user weight matrix W. This is the data Rachel's heatmap needs.

In [ ]:
import sys
sys.path.insert(0, '/content/LoRe')
import torch
from collections import defaultdict
device = 'cuda:0'

# Re-create the per-user embedding groupings the repo's training script uses
def group_embeddings_by_user(train_embeddings, test_embeddings, device):
    def process(dataset, seen_value, split_name):
        grouped = defaultdict(lambda: {'embeddings': []})
        for ex in dataset:
            info = ex.get('extra_info', {})
            if info.get('seen') == seen_value and info.get('split') == split_name:
                uid = info.get('user_id')
                if uid:
                    chosen = torch.tensor(info['chosen_conv_embedding'], dtype=torch.float32, device=device)
                    rejected = torch.tensor(info['rejected_conv_embedding'], dtype=torch.float32, device=device)
                    grouped[uid]['embeddings'].append(chosen - rejected)
        sorted_grouped, sorted_uids = [], []
        for uid in sorted(grouped.keys()):
            sorted_grouped.append(torch.stack(grouped[uid]['embeddings']))
            sorted_uids.append(uid)
        return sorted_grouped, sorted_uids
    train_seen, train_seen_uids = process(train_embeddings, True, 'train')
    train_unseen, train_unseen_uids = process(train_embeddings, False, 'train')
    test_seen, _ = process(test_embeddings, True, 'test')
    test_unseen, _ = process(test_embeddings, False, 'test')
    return (train_seen, train_seen_uids), (train_unseen, train_unseen_uids), test_seen, test_unseen

train_embeddings = torch.load('/content/LoRe/PRISM/data/prism/train_embeddings.pkl')
test_embeddings = torch.load('/content/LoRe/PRISM/data/prism/test_embeddings.pkl')
(train_seen, train_seen_uids), (train_unseen, train_unseen_uids), test_seen, test_unseen = group_embeddings_by_user(train_embeddings, test_embeddings, device)
print(f'Seen users: {len(train_seen)}  Unseen users: {len(train_unseen)}')

In [ ]:
# Extract V_sft from Skywork's final linear layer (the reference reward direction)
from transformers import AutoModel
rm = AutoModel.from_pretrained(
    'Skywork/Skywork-Reward-Llama-3.1-8B-v0.2',
    torch_dtype=torch.bfloat16, device_map=device,
    attn_implementation='eager', num_labels=1,
)
last_linear = None
for _, m in rm.named_modules():
    if isinstance(m, torch.nn.Linear):
        last_linear = m
V_sft = last_linear.weight[:, 0].to(device).to(torch.float32).reshape(-1, 1)
print('V_sft shape:', V_sft.shape)
del rm  # free VRAM for training
torch.cuda.empty_cache()

In [ ]:
# Train LoRe at K=8 on seen users, get V (basis matrix) and W (per-user simplex weights)
from utils import solve_regularized_simplex
K = 8
alpha = 1e4
W_real, V_basis = solve_regularized_simplex(
    V_sft, alpha, train_seen, num_basis_vectors=K,
    num_iterations=1000, learning_rate=0.01,
)
print('V_basis shape:', V_basis.shape)
print('W_real type:', type(W_real), 'len:', len(W_real) if hasattr(W_real, "__len__") else 'scalar')

## 9. Construct simulated extreme users and fit their weights on the frozen basis

For each basis k ∈ {0..7}, build a synthetic user whose preferences are perfectly aligned with that basis direction (chosen-minus-rejected embedding ≈ V_basis[:,k] + small noise). Fit their K-dim weight vector on the frozen V. If the math works, simulated user k should come out close to one-hot at position k.

In [ ]:
from utils import learn_multiple_few_shot

N_PAIRS_SIM = 30  # pairs per simulated user
NOISE_STD = 0.1   # relative to basis-direction norm

sim_users = []
sim_labels = []
for k in range(K):
    direction = V_basis[:, k].detach()
    direction = direction / (direction.norm() + 1e-8) * 50  # scale to roughly match real embedding-diff norms
    noise = NOISE_STD * direction.norm() * torch.randn(N_PAIRS_SIM, V_basis.shape[0], device=device)
    diffs = direction.unsqueeze(0).expand(N_PAIRS_SIM, -1) + noise
    sim_users.append(diffs)
    sim_labels.append(f'sim_V{k}')

# Add a few "mixed" simulated users: weighted combination of two bases
import random
random.seed(0)
for k1, k2 in [(0, 4), (2, 6), (1, 7)]:
    d1 = V_basis[:, k1].detach(); d1 = d1 / (d1.norm() + 1e-8) * 50
    d2 = V_basis[:, k2].detach(); d2 = d2 / (d2.norm() + 1e-8) * 50
    direction = 0.5 * d1 + 0.5 * d2
    noise = NOISE_STD * direction.norm() * torch.randn(N_PAIRS_SIM, V_basis.shape[0], device=device)
    diffs = direction.unsqueeze(0).expand(N_PAIRS_SIM, -1) + noise
    sim_users.append(diffs)
    sim_labels.append(f'sim_V{k1}+V{k2}')

W_sim = learn_multiple_few_shot(sim_users, V_basis, num_iterations=1000, learning_rate=0.01)
print('W_sim:', type(W_sim), getattr(W_sim, 'shape', None))

## 10. Heatmap: real users vs simulated users on the same bases

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def to_np_matrix(W):
    if isinstance(W, torch.Tensor):
        return W.detach().cpu().numpy()
    return np.stack([w.detach().cpu().numpy().reshape(-1) for w in W], axis=0)

W_real_np = to_np_matrix(W_real)
W_sim_np = to_np_matrix(W_sim)
print('W_real_np:', W_real_np.shape, 'W_sim_np:', W_sim_np.shape)

# Subsample real users for legibility
N_REAL_SHOW = min(40, W_real_np.shape[0])
rng = np.random.default_rng(0)
real_idx = rng.choice(W_real_np.shape[0], size=N_REAL_SHOW, replace=False)
W_real_show = W_real_np[real_idx]
real_labels = [f'real_{i}' for i in range(N_REAL_SHOW)]

full = np.concatenate([W_real_show, W_sim_np], axis=0)
row_labels = real_labels + sim_labels

fig, ax = plt.subplots(figsize=(7, max(8, 0.22 * len(row_labels))))
im = ax.imshow(full, aspect='auto', cmap='viridis', vmin=0, vmax=full.max())
ax.set_xticks(range(K))
ax.set_xticklabels([f'V_{i}' for i in range(K)])
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=7)
ax.axhline(y=N_REAL_SHOW - 0.5, color='red', linewidth=1.5)
ax.set_title(f'LoRe basis weights (K={K}): real PRISM users vs simulated extreme users')
plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
plt.tight_layout()
plt.savefig('/content/heatmap_lore_K8.png', dpi=160, bbox_inches='tight')
plt.show()
print('Saved /content/heatmap_lore_K8.png')

## 11. Quick interpretability summary

In [ ]:
print('Average weight per basis (across real users):')
for k in range(K):
    print(f'  V_{k}: {W_real_np[:, k].mean():.3f}  (max={W_real_np[:, k].max():.3f}, frac>0.5={float((W_real_np[:, k] > 0.5).mean()):.2f})')

popular = int(W_real_np.mean(axis=0).argmax())
print(f'\nMost popular basis among real users: V_{popular}')

print('\nSimulated user peak basis (should match construction):')
for label, row in zip(sim_labels, W_sim_np):
    top = int(row.argmax())
    print(f'  {label}: argmax=V_{top}  weight={row[top]:.3f}')